# Invoice Region Detection and Business Parameter Extraction Using CNN, SSD, IoU, OCR, and Streamlit

**Member:** Jordan
**Role:** Invoice Region Detection, SSD/CNN Object Detection, IoU Evaluation Lead

**Objective:** Detect invoice business regions (line items table, totals, payment terms, terms & conditions, reference numbers, seller/buyer info, etc.) using SSD-style object detection, and evaluate against ground truth using IoU.

**Inputs expected:**
- `data/processed/invoice_manifest.csv` (Rolando)
- `data/annotations/layout_bboxes.csv` (custom-annotated region boxes -- see `../../runbook.md`)
- `config/label_schema.json`

**Outputs generated:**
- `outputs/predictions/region_predictions.csv`
- `outputs/metrics/region_iou_metrics.json`
- `outputs/figures/region_detection_examples.png`
- `models/region_detector/`

> Run this notebook top-to-bottom in Google Colab, or locally with the repo's virtualenv.
> Paths are resolved via `src/config.py` (pathlib-based) — never hardcode absolute local paths.


In [ ]:
# --- Google Colab setup cell ---
# If running in Colab: clone the repo (or mount Drive if you cloned there already) and
# install dependencies. Safe to skip locally if the repo is already on disk with deps installed.

import sys, os

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/<your-org>/invoice-image-processing.git"  # TODO: set this
    REPO_DIR = "/content/invoice-image-processing"

    if not os.path.exists(REPO_DIR):
        os.system(f"git clone {REPO_URL} {REPO_DIR}")
    os.chdir(REPO_DIR)
    os.system("pip install -q -r requirements.txt")

    # Kaggle API credentials (upload kaggle.json when prompted) -- see dataset_sources.md
    from google.colab import files
    if not os.path.exists("/root/.kaggle/kaggle.json"):
        print("Upload your kaggle.json (Kaggle -> Account -> Create New API Token):")
        uploaded = files.upload()
        os.makedirs("/root/.kaggle", exist_ok=True)
        for fname in uploaded:
            os.replace(fname, "/root/.kaggle/kaggle.json")
        os.chmod("/root/.kaggle/kaggle.json", 0o600)

    print("Colab environment ready. Working directory:", os.getcwd())
else:
    print("Not running in Colab -- assuming local repo checkout with requirements installed.")


In [ ]:
# --- Dataset path setup cell ---
# All paths go through src.config.PATHS (pathlib-based, no hardcoded absolute paths).

import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "requirements.txt").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import PATHS, load_label_schema, load_required_fields

print("Repo root:", PATHS.repo_root)
print("Raw data dir:", PATHS.raw_dir)
print("Outputs dir:", PATHS.outputs_dir)

# If raw data isn't present yet, download it (see dataset_sources.md for kaggle.json setup):
#   python scripts/download_datasets.py --dataset all


In [ ]:
# --- Imports cell ---
import json

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from src.image_preprocessing import preprocess_pipeline, to_grayscale, resize_image, denoise_image, threshold_image, deskew_image
from src.visualization import draw_boxes, show_image_grid
from src.annotation_utils import load_annotations, boxes_for_image
from src.iou import compute_iou, precision_recall_iou, evaluate_predictions_df


## Concept note for the report/presentation

This notebook implements **SSD-style object detection**: a CNN backbone produces feature
maps at multiple scales, each cell predicts bounding-box offsets + class scores against a
set of default/anchor boxes, and Non-Max Suppression resolves overlapping predictions. If
full SSD training is too heavy for the timeline, a lightweight YOLOv8 (Ultralytics) model is
used as a documented **implementation fallback** -- but the underlying task (bounding-box
object detection, evaluated with IoU) and this explanation stay the same either way.

In [ ]:
from src.config import load_label_schema
REGION_LABELS = load_label_schema()["region_labels"]
print(REGION_LABELS)


## 1. Load processed images (Rolando) and region annotations

In [ ]:
from src.data_loader import load_manifest
from src.annotation_utils import load_annotations

manifest = load_manifest()

ann_path = PATHS.annotations_dir / "layout_bboxes.csv"
if ann_path.exists() and ann_path.stat().st_size > 0:
    region_annotations = load_annotations(ann_path, labels=REGION_LABELS)
else:
    region_annotations = pd.DataFrame(columns=["document_id","image_path","label","xmin","ymin","xmax","ymax","split","annotation_source"])
    print("No region annotations found yet -- see ../../runbook.md 'Manual annotation' section.")
region_annotations.head()


## 2. Priority regions for this project

In [ ]:
PRIORITY_REGIONS = [
    "reference_numbers_region",
    "line_items_table",
    "total_amount_region",
    "payment_terms_region",
    "terms_and_conditions_region",
    "seller_info",
    "buyer_info",
]
print(PRIORITY_REGIONS)


## 3. Train or demonstrate SSD-style region detection

In [ ]:
# TODO: train an SSD (e.g. torchvision.models.detection.ssd300_vgg16, fine-tuned) or,
# if timeline requires, the documented YOLOv8 fallback (ultralytics), on region_annotations.
#
# from ultralytics import YOLO
# model = YOLO("yolov8n.pt")
# model.train(data=<yaml derived from layout_bboxes.csv>, epochs=30)


## 4. Run inference, build predictions dataframe

In [ ]:
# TODO: run the trained detector over manifest images (or a sample) -> dataframe with
# columns: document_id, image_path, label, xmin, ymin, xmax, ymax, confidence
region_predictions = pd.DataFrame(columns=["document_id", "image_path", "label", "xmin", "ymin", "xmax", "ymax", "confidence"])


## 5. IoU evaluation (src/iou.py — the shared project implementation)

In [ ]:
from src.iou import evaluate_predictions_df

if not region_predictions.empty and not region_annotations.empty:
    region_iou_metrics = evaluate_predictions_df(region_predictions, region_annotations, REGION_LABELS, iou_threshold=0.5)
else:
    region_iou_metrics = {"per_label": {label: {"precision": None, "recall": None, "mean_iou": None} for label in REGION_LABELS}, "overall_mean_iou": None}

print(json.dumps(region_iou_metrics, indent=2))


## 6. Save example predictions drawn on images

In [ ]:
from src.visualization import draw_boxes, show_image_grid

example_imgs, example_titles = [], []
for doc_id in region_predictions["document_id"].unique()[:6]:
    pass  # TODO: img = cv2.imread(path from manifest); boxes = region_predictions rows for doc_id
    # example_imgs.append(draw_boxes(img, boxes)); example_titles.append(doc_id)

PATHS.figures_dir.mkdir(parents=True, exist_ok=True)
if example_imgs:
    show_image_grid(example_imgs, titles=example_titles, cols=3, save_path=PATHS.figures_dir / "region_detection_examples.png")
else:
    print("No example predictions yet -- fill in section 3/4 with a trained model first.")


In [ ]:
# --- Final export cell ---
# Save every output required by model_interface_contract.md

PATHS.predictions_dir.mkdir(parents=True, exist_ok=True)
region_predictions.to_csv(PATHS.predictions_dir / "region_predictions.csv", index=False)

PATHS.metrics_dir.mkdir(parents=True, exist_ok=True)
(PATHS.metrics_dir / "region_iou_metrics.json").write_text(json.dumps(region_iou_metrics, indent=2), encoding="utf-8")

(PATHS.models_dir / "region_detector").mkdir(parents=True, exist_ok=True)
# TODO: save actual trained weights into models/region_detector/.

member_out = PATHS.member_outputs_dir("jordan_region_iou")
member_out.mkdir(parents=True, exist_ok=True)
region_predictions.to_csv(member_out / "region_predictions.csv", index=False)
(member_out / "region_iou_metrics.json").write_text(json.dumps(region_iou_metrics, indent=2), encoding="utf-8")

print('Export complete.')
